# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinadh2314/srinadh-flyrank-intership/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*
## Feature vector

I use observable feature-window signals that are available before the prediction moment. The feature vector contains search and engagement signals from February 2026. Missing numeric values are filled with zero for consistent model input.

In [16]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully")

HF_TOKEN loaded successfully


In [17]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("HF_TOKEN loaded successfully")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Warehouse connection ready")

feature_query = """
SELECT
    client_hash_id,
    content_hash_id,
    AVG(gsc_impressions) AS avg_gsc_impressions,
    AVG(gsc_clicks) AS avg_gsc_clicks,
    AVG(gsc_avg_position) AS avg_gsc_avg_position,
    AVG(ga4_sessions) AS avg_ga4_sessions,
    AVG(scroll_events) AS avg_scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
"""

feature_df = con.execute(feature_query).df()

print("Feature dataframe shape:", feature_df.shape)


honest_features = [
    "avg_gsc_impressions",
    "avg_gsc_clicks",
    "avg_gsc_avg_position",
    "avg_ga4_sessions",
    "avg_scroll_events"
]

feature_vector = feature_df[
    ["client_hash_id", "content_hash_id"] + honest_features
].copy()

feature_vector[honest_features] = (
    feature_vector[honest_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

print("Feature vector shape:", feature_vector.shape)
print("Number of features:", len(honest_features))
print("Features:", honest_features)

display(feature_vector.head(10))

HF_TOKEN loaded successfully
Warehouse connection ready


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature dataframe shape: (321546, 7)
Feature vector shape: (321546, 7)
Number of features: 5
Features: ['avg_gsc_impressions', 'avg_gsc_clicks', 'avg_gsc_avg_position', 'avg_ga4_sessions', 'avg_scroll_events']


,client_hash_id,content_hash_id,avg_gsc_impressions,avg_gsc_clicks,avg_gsc_avg_position,avg_ga4_sessions,avg_scroll_events
0,client_e547b89c05043229,content_1eea820697c3b95a,10.678571,0.000000,12.946228,0.000000,0.000000
1,client_e547b89c05043229,content_9abd8b303f805847,26.178571,0.214286,6.495085,0.214286,0.000000
2,client_e547b89c05043229,content_5f58c55cbfee172a,18.357143,0.000000,10.490023,0.035714,0.000000
3,client_e547b89c05043229,content_6fe390ba3af1e456,104.678571,0.107143,38.436254,0.214286,0.035714
4,client_e547b89c05043229,content_3ad5d2160242b9ca,34.642857,0.071429,9.710810,0.107143,0.035714
5,client_e547b89c05043229,content_a2bd730a7cf68316,19.678571,0.035714,6.017373,0.107143,0.000000
6,client_e547b89c05043229,content_cbe43d4b6ce2d320,10.392857,0.000000,4.333115,0.107143,0.000000
7,client_e547b89c05043229,content_babd931911c9ee33,95.714286,1.107143,5.046562,0.857143,0.178571
8,client_e547b89c05043229,content_9c36ace83c73b5eb,14.857143,0.035714,40.800604,0.035714,0.000000
9,client_e547b89c05043229,content_431784c057b25a5d,130.035714,0.214286,8.784133,0.642857,0.000000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## Feature notes

- `avg_gsc_impressions`: Average Google Search Console impressions during the February 2026 feature window. Missing values are filled with 0. This is available before the March outcome window.

- `avg_gsc_clicks`: Average Google Search Console clicks during the February 2026 feature window. Missing values are filled with 0. This is available before the prediction moment.

- `avg_gsc_avg_position`: Average Google Search Console search position during the February 2026 feature window. Missing values are filled with 0. This is available before the March outcome window.

- `avg_ga4_sessions`: Average GA4 sessions during the February 2026 feature window. Missing values are filled with 0. This is available before the prediction moment.

- `avg_scroll_events`: Average scroll events during the February 2026 feature window. Missing values are filled with 0. This is available before the prediction moment.

No March outcome field is included in the feature vector.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature types, missing values, and whether all model features are present.

feature_notes = pd.DataFrame({
    "feature": honest_features,
    "dtype": [feature_vector[f].dtype for f in honest_features],
    "missing_after_fill": [
        feature_vector[f].isna().sum()
        for f in honest_features
    ]
})

display(feature_notes)

print(
    "All selected features available in feature vector:",
    all(f in feature_vector.columns for f in honest_features)
)

print(
    "March outcome field included:",
    "march_impressions" in feature_vector.columns
)


,feature,dtype,missing_after_fill
0,avg_gsc_impressions,float64,0
1,avg_gsc_clicks,float64,0
2,avg_gsc_avg_position,float64,0
3,avg_ga4_sessions,float64,0
4,avg_scroll_events,float64,0


All selected features available in feature vector: True
March outcome field included: False


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*
## The leakage hunt

I checked the feature vector for label-derived fields, future-window fields, and product decision flags. The honest feature vector uses only February 2026 observable signals. March outcome information must not be used as a predictive feature because it would not be available at the decision moment.

In [19]:


forbidden_features = [
    "march_impressions",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "future_impressions",
    "future_clicks"
]

leakage_check = pd.DataFrame({
    "field": forbidden_features,
    "present_in_feature_vector": [
        field in feature_vector.columns
        for field in forbidden_features
    ]
})

display(leakage_check)

print(
    "Number of forbidden fields in feature vector:",
    leakage_check["present_in_feature_vector"].sum()
)


,field,present_in_feature_vector
0,march_impressions,False
1,trend_direction,False
2,trend_pct,False
3,is_declining_label,False
4,future_impressions,False
5,future_clicks,False


Number of forbidden fields in feature vector: 0


In [20]:


label_query = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
"""

label_df = con.execute(label_query).df()

print("March outcome rows:", len(label_df))
print("March outcome field available:", "march_impressions" in label_df.columns)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March outcome rows: 331437
March outcome field available: True


In [21]:


leaky_feature_vector = feature_vector.copy()

march_label = label_df[
    ["client_hash_id", "content_hash_id", "march_impressions"]
].copy()

leaky_feature_vector = leaky_feature_vector.merge(
    march_label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(
    "March outcome deliberately added:",
    "march_impressions" in leaky_feature_vector.columns
)

print(
    "Leaky feature vector shape:",
    leaky_feature_vector.shape
)

March outcome deliberately added: True
Leaky feature vector shape: (321546, 8)


In [22]:
honest_feature_vector = leaky_feature_vector.drop(
    columns=["march_impressions"]
)

print(
    "Leaked feature removed:",
    "march_impressions" not in honest_feature_vector.columns
)


Leaked feature removed: True


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
## What I excluded and why

- `march_impressions`: Excluded because it comes from the future outcome window and would leak information that is not available at the prediction moment.

- `trend_direction`: Excluded because it represents an outcome/trend field rather than a pre-decision feature for this feature vector.

- `trend_pct`: Excluded because it is derived from outcome movement and could introduce label or future-window information.

- `is_declining_label`: Excluded because it is a label-derived field and should not be used as a predictive feature.

- `future_impressions`: Excluded because it contains future-window information.

- `future_clicks`: Excluded because it contains future-window information.

- Client and content identifiers: Used only for grouping and joining; excluded from the predictive feature matrix because they are identifiers rather than behavioral features.

In [23]:

excluded_fields = [
    "march_impressions",
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "future_impressions",
    "future_clicks",
    "client_hash_id",
    "content_hash_id"
]

excluded_reasons = pd.DataFrame({
    "field": excluded_fields,
    "excluded": [True] * len(excluded_fields)
})

display(excluded_reasons)

print("Predictive features kept:")
print(honest_features)


,field,excluded
0,march_impressions,True
1,trend_direction,True
2,trend_pct,True
3,is_declining_label,True
4,future_impressions,True
5,future_clicks,True
6,client_hash_id,True
7,content_hash_id,True


Predictive features kept:
['avg_gsc_impressions', 'avg_gsc_clicks', 'avg_gsc_avg_position', 'avg_ga4_sessions', 'avg_scroll_events']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.